# 기업 이름 통일

DART(공시)와 뉴스에 나온 **기업 이름을 하나의 기업 노드로 묶고 고정 ID를 붙인다**(`graphDB.md` 결정할 것 #16, 6장). 그래프에서 같은 회사가 여러 노드로 나뉘면 연결 경로가 끊기기 때문에, 이 ID가 그래프의 기초가 된다.

- **국내 상장사**: 서비스 대상 후보(DART 기업 목록 1,067개)는 종목코드로 `KR:종목코드`를 붙인다. 한글명·영문명과 같은 이름은 이 ID로 모인다.
- **후보 밖 상장사**(`sector.json`의 나머지 1,588개)도 `KR:종목코드`를 붙이되, 이름이 같은 다른 회사가 붙지 않게 조건을 둔다(4절 1-2, 3-2, 4-2). 같은 회사가 한글 이름으로는 `KR:`, 영어 이름으로는 `EN:` 노드로 갈리지 않게 하려는 것이다.
- **그 밖의 기업**(국내 비상장사, 해외 기업): 정규화한 영어 이름으로 `EN:이름`, 영어 이름이 없으면 정규화한 한글 이름으로 `KO:이름`을 붙인다.
- 같은 회사의 여러 이름을 잇는 근거:
  1. 정규화한 이름이 같다("Samsung Electronics Co., Ltd." = "SAMSUNG ELECTRONICS CO,.LTD").
  2. 한 언급에 한글 이름과 영어 이름이 함께 적혀 있다(DART 거래 상대의 `name`과 `name_en`, 이름 확신 `name_sure`가 true일 때만).
  3. 뉴스 검색어의 약칭("TSMC")이 전체에서 한 회사만 가리킨다.
- **그룹 이름만 적힌 것**("Samsung", "LG", "Hyundai")은 어느 회사인지 알 수 없어 묶지 않고 `애매` 표시를 한다.
- 서로 다른 두 상장사가 한 덩어리로 묶이게 되는 연결은 하지 않고 기록한다.
- 규칙으로 하는 첫 단계다. 실제 v5 데이터로 그래프를 만들 때 남은 애매한 이름과 한글·영어가 이어지지 않은 이름을 LLM으로 보완할 수 있다.
- 서버나 LLM에 요청하지 않는다.

**입력**

| 파일 | 쓰는 것 |
|---|---|
| `source/dart/parsed/companies.csv` | 서비스 대상 후보 1,067개: 종목코드, 종목명, 법인명, 영문 법인명 |
| `../market_data/sector.json` | 상장사 2,655개의 종목명(후보 밖 상장사의 한글 이름) |
| `source/dart/raw/corpCode/corpCode_*.zip` | DART 고유번호 목록(`dart_collector`가 받음): 상장사의 법인명·영어 이름 |
| `source/dart/parsed/subsidiaries.csv`, `associates.csv`, `investments.csv`, `shareholders.csv` | 지분 관계 상대(종속회사, 관계기업, 출자 대상, 법인 주주) |
| `source/dart/business/final/dart_business_v5.json` | 거래 상대: 매입처, 고객, 경쟁사, 계약 상대, 기타 관계 상대 |
| `source/news/final/news_final.json` | 뉴스 개체 중 기업 |

**출력** (`source/graph/`)

| 파일 | 한 줄 |
|---|---|
| `companies.csv` | 기업 노드 하나: ID, 대표 이름(영어·한글), 종목코드, 나라, 언급 수, 다른 이름들 |
| `company_mentions.csv` | 언급 하나: 출처, 문서, 자리(매입처·뉴스 개체 등), 원래 이름, 붙은 ID, 묶은 근거 |
| `company_ambiguous.csv` | 묶지 못한 그룹 이름 언급 |

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `dart_business_cleaner`·`news_cleaner` → **`name_unifier`** → `product_dictionary` → `graph_db` |
| 외부 요청·비용 | 없음 |
| 다시 실행 | 언제든 된다. 결과 파일은 내용이 바뀔 때만 다시 쓴다 |
| 처리 순서 | 이름 키 만들기 → 언급 모으기(DART 지분 표·사업의 내용, 뉴스 개체) → 같은 회사끼리 묶기(상장사 이름, 한 언급의 두 이름, 약칭, 자회사 떼기) → 기업 ID 정하기 → 저장 → 확인 |
| 결과 | `source/graph/companies.csv`(기업 노드), `company_mentions.csv`(언급 → 기업 ID), `company_ambiguous.csv`(그룹 이름만 적힌 언급) |

## 1. 설정

In [1]:
# ── 설정: 폴더 ──
import collections
import json
import re
import unicodedata
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'
PARSED = ROOT / 'source' / 'dart' / 'parsed'
OUT = ROOT / 'source' / 'graph'
MARKET = ROOT.parent / 'market_data'
# dart_collector가 받은 DART 고유번호 목록 중 가장 최근 것(상장사의 법인명·영어 이름)
CORPCODE = sorted((ROOT / 'source' / 'dart' / 'raw' / 'corpCode').glob('corpCode_*.zip'))[-1]
OUT.mkdir(parents=True, exist_ok=True)

## 2. 이름 정규화

**영어 이름** (`norm_en`)
- 소문자, `&` → and, 문장부호를 공백으로, 악센트 제거(Wärtsilä → wartsila).
- 맨 앞 "the"와 끝의 법인 표기를 지운다: inc, corp, corporation, co, company, ltd, limited, llc, plc, sa, ag, nv, se, spa, gmbh, bv, ab, asa, oyj, kk, pte, pty, srl, sas, lp.
- 법인 표기를 지운 결과가 **그룹 이름**(samsung, lg, sk, hyundai 등)만 남으면 지우지 않는다. "LG Corp."는 지주회사 ㈜LG이고, "LG"만 적힌 것은 그룹 전체일 수 있어 둘을 가른다.

**한글 이름** (`norm_ko`): (주)·㈜·주식회사·(유)·유한회사와 띄어쓰기·문장부호를 지우고, 안의 영어는 소문자로.

In [2]:
# ── 이름 키: 같은 회사면 같은 키가 나오게 법인 표기·문장부호를 지운다 ──
# 이름 끝에서 지울 법인 표기
SUFFIXES = {'inc', 'incorporated', 'corp', 'corporation', 'co', 'company', 'ltd', 'limited', 'llc', 'plc', 'sa', 'ag', 'nv', 'se',
            'spa', 'gmbh', 'bv', 'ab', 'asa', 'oyj', 'kk', 'pte', 'pty', 'srl', 'sas', 'lp', 'ltda', 'sab', 'cv', 'de'}
# 그룹 이름: 이것만으로는 어느 계열사인지 알 수 없다(애매로 둔다)
GROUP_STEMS = {'samsung', 'lg', 'sk', 'hyundai', 'hd hyundai', 'lotte', 'hanwha', 'doosan', 'gs', 'cj', 'posco', 'hyosung',
               'kolon', 'ls', 'kumho', 'hanjin', 'shinsegae', 'kt', 'mitsubishi', 'mitsui', 'sumitomo', 'marubeni'}
HANGUL = re.compile('[가-힣]')


def norm_en(name):
    """영어 회사 이름을 비교용으로 정규화한다(악센트·문장부호·끝의 법인 표기 제거). 그룹 이름만 남으면 법인 표기 한 단어를 살린다."""
    s = unicodedata.normalize('NFKD', name or '').encode('ascii', 'ignore').decode()
    s = s.lower().replace('&', ' and ')
    s = re.sub(r"[^a-z0-9 ]+", ' ', s)
    toks = s.split()
    if toks[:1] == ['the']:
        toks = toks[1:]
    full = list(toks)
    while len(toks) > 1 and toks[-1] in SUFFIXES:
        toks = toks[:-1]
    base = ' '.join(toks)
    if base in GROUP_STEMS and len(full) > len(toks):     # "LG Corp." → "lg corp"
        return ' '.join(full[:len(toks) + 1])
    return base


def norm_ko(name):
    """한글 회사 이름에서 (주)·주식회사 같은 표기와 공백·문장부호를 지운다."""
    s = re.sub(r'\(주\)|㈜|주식회사|\(유\)|유한회사|\(株\)|株式會社', '', name or '')
    return re.sub(r'[\s.,·\-()\[\]&/]', '', s).lower()


def keys_of(name):
    """이름 하나의 키: 한글이 있으면 ko:, 없으면 en:."""
    if not name or not name.strip():
        return None
    return 'ko:' + norm_ko(name) if HANGUL.search(name) else 'en:' + norm_en(name)


# 정규화 예시
for n in ['Samsung Electronics Co., Ltd.', 'SAMSUNG ELECTRONICS CO,.LTD', 'LG Corp.', 'LG', 'Wärtsilä Corporation',
          'The Kroger Co.', '삼성전기㈜', '(주)한익스프레스', 'SK hynix Inc.']:
    print(f'{n!r:35} → {keys_of(n)}')

'Samsung Electronics Co., Ltd.'     → en:samsung electronics
'SAMSUNG ELECTRONICS CO,.LTD'       → en:samsung electronics
'LG Corp.'                          → en:lg corp
'LG'                                → en:lg
'Wärtsilä Corporation'              → en:wartsila
'The Kroger Co.'                    → en:kroger
'삼성전기㈜'                             → ko:삼성전기
'(주)한익스프레스'                         → ko:한익스프레스
'SK hynix Inc.'                     → en:sk hynix


## 3. 언급 모으기

기업이 나오는 모든 자리를 한 표로 모은다. 한 줄이 언급 하나다.

| 출처 | 자리 | 이름 칸 |
|---|---|---|
| DART 지분 표 | 종속회사, 관계기업, 출자 대상, 법인 주주 | `name` (영어 또는 한글) |
| DART 사업의 내용 | 매입처, 고객, 경쟁사, 계약 상대, 기타 관계 상대 | `name`(원문), `name_en`, `name_sure` |
| 뉴스 | 개체 중 종류가 기업인 것 | `name`(기사 표기), `name_en`, `search_terms`, `country` |

In [3]:
# ── 언급 모으기: 회사 이름이 나온 곳을 한 표(M)로 ──
companies = pd.read_csv(PARSED / 'companies.csv', dtype=str).fillna('')
rows = []


def add(source, doc, field, name, name_en='', sure=True, country='', aliases=(), ref=''):
    """회사 언급 한 줄을 더한다. sure는 LLM이 이름을 확신했는지, aliases는 기사 속 다른 이름(검색어)."""
    rows.append(dict(source=source, doc=doc, field=field, name=(name or '').strip(), name_en=(name_en or '').strip(),
                     sure=sure, country=country or '', aliases=list(aliases), ref=ref))


# DART 지분 표
for f, field in [('subsidiaries', '종속회사'), ('associates', '관계기업'), ('investments', '출자 대상')]:
    t = pd.read_csv(PARSED / f'{f}.csv', dtype=str).fillna('')
    for r in t.itertuples():
        add('dart_지분', r.stock_code, field, r.name, country=getattr(r, 'country', ''))
sh = pd.read_csv(PARSED / 'shareholders.csv', dtype=str).fillna('')
for r in sh[sh['is_corp'] == 'True'].itertuples():
    add('dart_지분', r.stock_code, '법인 주주', r.name)

# DART 사업의 내용
biz = json.loads((ROOT / 'source' / 'dart' / 'business' / 'final' / 'dart_business_v5.json').read_text(encoding='utf-8'))['companies']
for code_, c in biz.items():
    for m in c['materials']:
        for s in m['suppliers']:
            add('dart_사업', code_, '매입처', s['name'], s['name_en'], s['name_sure'])
    for x in c['customers']:
        add('dart_사업', code_, '고객', x['name'], x['name_en'], x['name_sure'])
    for x in c['competitors']:
        add('dart_사업', code_, '경쟁사', x['name'], x['name_en'], x['name_sure'])
    for x in c['contracts']:
        add('dart_사업', code_, '계약 상대', x['counterparty'], x['name_en'], x['name_sure'])
    for x in c['other_relations']:
        add('dart_사업', code_, '기타 관계 상대', x['counterparty'], x['name_en'], True)

# 뉴스 개체(기업)
news = json.loads((ROOT / 'source' / 'news' / 'final' / 'news_final.json').read_text(encoding='utf-8'))['articles']
for aid, o in news.items():
    for e in o['entities']:
        if e['type'] == '기업':
            add('news', aid, '뉴스 개체', e['name'], e['name_en'], True, e['country'], e['search_terms'], e['id'])

M = pd.DataFrame(rows)
M = M[(M['name'] != '') | (M['name_en'] != '')].reset_index(drop=True)
print(f'언급 {len(M):,}개'); display(M.groupby(['source', 'field']).size().rename('언급 수').reset_index())

언급 70,607개


,source,field,언급 수
0,dart_사업,경쟁사,1688
1,dart_사업,계약 상대,3526
2,dart_사업,고객,2979
3,dart_사업,기타 관계 상대,946
4,dart_사업,매입처,4045
5,dart_지분,관계기업,8357
6,dart_지분,법인 주주,1603
7,dart_지분,종속회사,19236
8,dart_지분,출자 대상,19936
9,news,뉴스 개체,8291


## 4. 묶기

이름 키(`ko:…`, `en:…`)와 상장사 ID(`KR:…`)를 점으로 보고, 같은 회사라는 근거가 있는 점끼리 잇는다(union-find). 이어진 덩어리 하나가 기업 하나다.

1. **상장사**: `KR:종목코드`를 영문 법인명·종목명·법인명의 키와 잇는다.
2. **한 언급의 두 이름**: 한 언급에 원문 이름과 영어 이름이 함께 있으면 잇는다(DART는 `name_sure`가 true일 때만). 단 원문 이름이 5자 이하 약칭이면(예: 기사에 "ADI"로 적힘) 3)으로 넘긴다.
3. **약칭**: 뉴스 검색어 중 약칭(대문자 2~6자, 예: TSMC)이나 두 단어 이상인 것, 기사에 약칭으로 적힌 이름은 **전체에서 한 회사만 가리킬 때만** 그 회사와 잇는다. "ADI"는 Analog Devices와 Acciaierie d'Italia가, "AV"는 AeroVironment와 Alumni Ventures가 같이 써서 잇지 않는다.

1-2. **후보 밖 상장사**: 한글 이름(종목명, DART 법인명)은 3자 이상이면 잇는다. 국내 공시·기사의 한글 표기라 국내 회사로 본다. 두 글자 이하(삼진, 서한)는 다른 회사일 수 있어 잇지 않는다. 영어 이름(DART 고유번호 목록)은 바로 잇지 않고 3-2)로 넘긴다. 5자 이하 영어 이름(SEC, DSM 등)과 상장사끼리 같은 영어 이름(SG, ALT)은 쓰지 않는다.
3-2. **후보 밖 상장사의 영어 이름**: 언급마다 나라를 보고 잇는다. DART 언급은 나라가 해외로 적히지 않았으면, 뉴스 언급은 나라가 한국으로 적혔을 때만 잇는다. 중국 자회사 "SHOWBOX Inc."나 태국 기업 "Igloo"가 국내 상장사(쇼박스, 이글루)에 붙지 않게 하려는 것이다.
3-3. **한글 이름이 상장사와 같은데 영어 이름 쪽은 상장사가 아니면 한글 이름을 따른다.** 공시의 "에이텍"에 LLM이 확신 없이 "ATEC"을 붙여 상장사와 이어지지 않던 경우다. 영어 이름이 그룹 이름(Hyosung, Hanwha)이면 지금처럼 애매로 둔다.
- 1-2의 예외: 대상 기업 이름에서 그룹 이름을 떼면 후보 밖 상장사 이름과 같아지는 경우(현대모비스 → 모비스)는 약칭일 수 있어 그 이름으로 잇지 않는다.
4-2. **해외로 적힌 언급이 후보 밖 상장사로 묶였으면 떼어 낸다**(한글 이름이 이어 준 경우). 법인 표기를 살린 이름으로 따로 노드를 만든다. 해외에 본사를 둔 상장사(종목코드 9…, 코오롱티슈진 등)와 국내 법인 표기((주)·㈜·주식회사)가 있는 이름은 그대로 둔다.

- 그룹 이름 키(`en:samsung` 등)는 잇지 않는다.
- 서로 다른 상장사 둘이 한 덩어리가 되는 연결은 하지 않고 `conflicts`에 남긴다.
- **자기 지분 표의 자회사는 자기 회사로 묶지 않는다.** 법인 표기를 무시하고 비교하다 보니 모회사와 이름이 같은 자회사(삼성전자의 "Samsung Electronics GmbH", 지주사 동아제약의 "동아제약(주)")가 모회사 노드로 합쳐졌다. 회사 자신의 종속회사·관계기업·출자 대상 표에 나온 이름이 그 회사로 묶이면, 법인 표기를 살린 이름으로 따로 노드를 만든다(`EN:samsung electronics gmbh`). 법인 표기 없이 모회사와 똑같이 적혔으면(신도리코의 "SINDOH") 이름 뒤에 "(종목코드 자회사)"를 붙인다. 자기 지분 표에 자기 자신이 자회사로 나올 수는 없으므로 다른 경우에는 영향이 없다. 법인 주주 표의 자기 회사는 자기주식이라 그대로 둔다.

In [4]:
# ── 같은 회사끼리 묶기(union-find). 상장사 둘이 한 덩어리가 되는 묶음은 하지 않는다 ──
# parent: union-find 부모, kr_of[대표] = 그 덩어리의 상장사 ID
parent, kr_of = {}, {}


def find(x):
    """x가 속한 덩어리의 대표를 찾는다(경로 압축)."""
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


conflicts = []


def union(a, b, why):
    """a와 b를 한 덩어리로 묶는다. 서로 다른 상장사가 섞이면 묶지 않고 conflicts에 남긴다."""
    if not a or not b:
        return
    ra, rb = find(a), find(b)
    if ra == rb:
        return
    ka, kb = kr_of.get(ra), kr_of.get(rb)
    if ka and kb and ka != kb:
        conflicts.append(dict(a=a, b=b, kr_a=ka, kr_b=kb, why=why))
        return
    parent[rb] = ra
    if kb and not ka:
        kr_of[ra] = kb


is_stem = lambda k: k and k.startswith('en:') and k[3:] in GROUP_STEMS
is_short = lambda k: k and k.startswith('en:') and len(k[3:].replace(' ', '')) <= 5   # 약칭(ADI, AV, MS 등)은 여러 회사가 같이 쓴다
alias_owner = collections.defaultdict(set)

# 1) 상장사
for r in companies.itertuples():
    kid = 'KR:' + r.stock_code
    find(kid)
    kr_of[kid] = kid
    for n in (r.corp_name_eng, r.stock_name, r.corp_name):
        k = keys_of(n)
        if k and not is_stem(k):
            union(kid, k, '상장사 이름')

# 1-2) 후보 밖 상장사(sector.json 2,655개 중 companies.csv에 없는 것): 이름이 같은 다른 회사가 붙지 않게 조건을 둔다
#   - 한글 이름(종목명·DART 법인명): 3자 이상이면 잇는다. 두 글자 이하(삼진, 서한)는 다른 회사일 수 있다
#   - 영어 이름(DART 고유번호 목록): 바로 잇지 않고 OUT_EN에 적어 두었다가 3-2)에서 언급마다 나라를 보고 잇는다.
#     5자 이하(SEC, DSM)와 상장사끼리 같은 영어 이름(SG, ALT)은 쓰지 않는다
stocks = json.loads((MARKET / 'sector.json').read_text(encoding='utf-8'))['stocks']
with zipfile.ZipFile(CORPCODE) as z:
    corp_xml = ET.fromstring(z.read(z.namelist()[0]))
corp_by_stock = {}                                   # 종목코드 → (DART 법인명, 영어 이름)
for e in corp_xml.iter('list'):
    sc = (e.findtext('stock_code') or '').strip()
    if sc:
        corp_by_stock[sc] = ((e.findtext('corp_name') or '').strip(), (e.findtext('corp_eng_name') or '').strip())
OUTSIDE = {c: v['security_name'] for c, v in stocks.items() if c not in set(companies['stock_code'])}
TARGET_KR = {'KR:' + c for c in companies['stock_code']}
en_count = collections.Counter('en:' + norm_en(corp_by_stock.get(c, ('', ''))[1]) for c in stocks)
OUT_EN = {}                                          # 영어 이름 키 → 후보 밖 상장사 ID
# 그룹 이름을 뗀 대상 기업 이름과 같은 후보 밖 상장사 이름은 약칭일 수 있어 한글로 잇지 않는다(공시의 "모비스"는 현대모비스)
GROUP_KO = ('HD현대', '현대', 'SK', 'LG', '삼성', '롯데', '한화', '두산', 'GS', 'CJ', '포스코', '효성', '코오롱', 'LS', '금호', '한진', '신세계', 'KT')
target_ko = {norm_ko(n) for n in companies['stock_name']}
ABBREV = {norm_ko(n) for n in OUTSIDE.values() if any(norm_ko(g) + norm_ko(n) in target_ko for g in GROUP_KO)}
for c, name in OUTSIDE.items():
    kid = 'KR:' + c
    find(kid)
    kr_of[kid] = kid
    corp_name, eng = corp_by_stock.get(c, ('', ''))
    for n in (name, corp_name):
        k = keys_of(n)
        if k and k.startswith('ko:') and len(k) - 3 >= 3 and k[3:] not in ABBREV:
            union(kid, k, '후보 밖 상장사 한글 이름')
    k = 'en:' + norm_en(eng) if eng else None
    if k and not is_stem(k) and not is_short(k) and en_count[k] == 1:
        OUT_EN[k] = kid
print(f'후보 밖 상장사 {len(OUTSIDE):,}개 / 영어 이름을 쓸 수 있는 것 {len(OUT_EN):,}개 / 약칭일 수 있어 한글로 잇지 않는 이름 {sorted(ABBREV)}')

# 2) 한 언급의 두 이름
M['key_name'] = M['name'].map(keys_of)
M['key_en'] = M['name_en'].map(lambda n: 'en:' + norm_en(n) if n and not HANGUL.search(n) else None)
for r in M.itertuples():
    for k in (r.key_name, r.key_en):
        if k:
            find(k)
    if not (r.sure and r.key_name and r.key_en) or is_stem(r.key_name) or is_stem(r.key_en):
        continue
    if is_short(r.key_name) and r.key_name != r.key_en:
        alias_owner[r.key_name].add(r.key_en)       # 약칭으로 적힌 이름은 3)에서 한 회사만 가리킬 때만 잇는다
    else:
        union(r.key_name, r.key_en, '한 언급의 두 이름')

# 3) 약칭: 전체에서 한 회사(덩어리)만 가리킬 때(기사에 약칭으로 적힌 이름 포함)
for r in M[M['source'] == 'news'].itertuples():
    owner = r.key_en or r.key_name
    for a in r.aliases:
        a = a.strip()
        if re.fullmatch(r'[A-Z][A-Z0-9&]{1,5}', a) or len(a.split()) >= 2:
            k = 'en:' + norm_en(a)
            if not is_stem(k) and k != owner:
                alias_owner[k].add(find(owner))
alias_used = 0
for k, owners in alias_owner.items():
    owners = {find(o) for o in owners}
    if len(owners) == 1:
        union(next(iter(owners)), k, '약칭')
        alias_used += 1
print(f'잇지 못한 상장사 충돌 {len(conflicts)}건, 약칭으로 이은 이름 {alias_used}개')

# 3-2) 후보 밖 상장사의 영어 이름: 국내 회사로 볼 수 있는 언급만 잇는다
#   DART 언급은 나라가 해외로 적히지 않았으면, 뉴스 언급은 나라가 한국으로 적혔을 때만.
#   해외 자회사("SHOWBOX Inc.", 중국)나 이름이 같은 해외 기업("Igloo", 태국)이 국내 상장사에 붙지 않게 한다
KOREA = {'대한민국', '한국', 'South Korea', 'Korea', 'Republic of Korea'}
is_abroad = lambda c: bool(c) and c not in KOREA
out_en_used = 0
for i, r in M.iterrows():
    k = r['key_en'] or r['key_name']
    if k not in OUT_EN or kr_of.get(find(k)):        # 이미 상장사로 이어진 덩어리면 그대로 둔다
        continue
    if (r['country'] in KOREA) if r['source'] == 'news' else not is_abroad(r['country']):
        kid = OUT_EN[k]
        union(kid, 'kr:' + kid[3:], '후보 밖 상장사 영어 이름')
        M.at[i, 'key_en'] = 'kr:' + kid[3:]           # 이 언급만 상장사로 옮긴다(같은 영어 이름의 다른 언급은 그대로)
        out_en_used += 1
print(f'후보 밖 상장사 영어 이름으로 이은 언급 {out_en_used}개')

# 3-3) 한글 이름이 상장사와 같은데 영어 이름 쪽은 상장사가 아니면 한글 이름을 따른다(영어 이름이 그룹 이름이면 애매로 둔다)
#   공시의 "에이텍"에 LLM이 확신 없이 "ATEC"을 붙여, 영어 이름 쪽 노드로 가서 상장사와 이어지지 않던 경우
ko_first = collections.Counter()
for i, r in M.iterrows():
    kn, ke = r['key_name'], r['key_en']
    if ke and kn and kn.startswith('ko:') and not is_stem(ke) and kr_of.get(find(kn)) and not kr_of.get(find(ke)) and not is_abroad(r['country']):
        M.at[i, 'key_en'] = None
        ko_first['후보' if kr_of[find(kn)] in TARGET_KR else '후보 밖'] += 1
print('한글 이름을 따라 상장사로 이은 언급', dict(ko_first))


# 4) 자기 지분 표의 자회사가 자기 회사로 묶인 경우: 법인 표기를 살린 키로 떼어 낸다
def full_key(name):
    if HANGUL.search(name):
        return 'ko:' + re.sub(r'[\s.,·\-\[\]&/]', '', name).lower()        # (주)·㈜를 지우지 않는다
    s_ = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode().lower().replace('&', ' and ')
    return 'en:' + ' '.join(re.sub(r'[^a-z0-9 ]+', ' ', s_).split())        # 법인 표기를 지우지 않는다


OWN_FIELDS = {'종속회사', '관계기업', '출자 대상'}
split_sub = []
for i, r in M[(M['source'] == 'dart_지분') & M['field'].isin(OWN_FIELDS)].iterrows():
    k = r['key_en'] or r['key_name']
    if k and kr_of.get(find(k)) == 'KR:' + r['doc']:
        fk = full_key(r['name'])
        if fk in parent and kr_of.get(find(fk)) == 'KR:' + r['doc']:     # 법인 표기 없이 모회사와 똑같이 적힘(신도리코의 "SINDOH")
            fk = f"{fk} ({r['doc']} 자회사)"
        M.at[i, 'key_name'], M.at[i, 'key_en'] = fk, None
        split_sub.append((r['doc'], r['field'], r['name']))
print(f'자기 회사로 묶였던 자회사 언급 {len(split_sub)}개를 따로 뗌(회사 {len({d for d, _, _ in split_sub})}곳)')

# 4-2) 해외로 적힌 언급이 후보 밖 상장사로 묶였으면 떼어 낸다(한글 이름이 이어 준 경우)
#   단 해외에 본사를 둔 상장사(종목코드 9…, 코오롱티슈진 등)와 국내 법인 표기((주)·㈜·주식회사)가 있는 이름은 그대로 둔다
KO_CORP = re.compile(r'\(주\)|㈜|주식회사')
split_abroad = []
for i, r in M.iterrows():
    k = r['key_en'] or r['key_name']
    kr = kr_of.get(find(k)) if k else None
    if kr and kr not in TARGET_KR and is_abroad(r['country']) and not kr.startswith('KR:9') and not KO_CORP.search(r['name']):
        fk = full_key(r['name'] or r['name_en'])
        if fk in parent and kr_of.get(find(fk)) == kr:
            fk = f"{fk} ({r['country']})"
        M.at[i, 'key_name'], M.at[i, 'key_en'] = fk, None
        split_abroad.append((r['source'], r['name'], r['country'], kr))
print(f'후보 밖 상장사로 묶였던 해외 언급 {len(split_abroad)}개를 따로 뗌')

후보 밖 상장사 1,588개 / 영어 이름을 쓸 수 있는 것 1,347개 / 약칭일 수 있어 한글로 잇지 않는 이름 ['cs', '모비스', '힘스']
잇지 못한 상장사 충돌 5건, 약칭으로 이은 이름 1521개


후보 밖 상장사 영어 이름으로 이은 언급 43개


한글 이름을 따라 상장사로 이은 언급 {'후보': 10, '후보 밖': 16}


자기 회사로 묶였던 자회사 언급 60개를 따로 뗌(회사 29곳)


후보 밖 상장사로 묶였던 해외 언급 1개를 따로 뗌


## 5. 기업 ID 정하기

- 덩어리에 상장사가 있으면 `KR:종목코드`.
- 없으면 덩어리 안에서 가장 많이 언급된 영어 키로 `EN:이름`, 영어 키가 없으면 `KO:이름`.
- 그룹 이름만 적힌 언급(`en:samsung` 등)은 ID를 붙이지 않고 `애매`로 둔다.
- 대표 이름: 상장사는 DART의 종목명·영문 법인명, 그 밖에는 가장 많이 쓰인 원래 표기.

In [5]:
# ── 기업 ID 정하기: 덩어리마다 KR:종목코드 > EN:가장 많이 쓰인 영어 이름 > KO:한글 이름 ──
# 언급의 대표 키: 영어 이름이 있으면 그것, 없으면 원래 이름
M['key'] = M['key_en'].where(M['key_en'].notna(), M['key_name'])
# 그룹 이름만 적힌 언급은 어느 회사에도 잇지 않는다
M['ambiguous'] = M['key'].map(is_stem)
M['root'] = M['key'].map(lambda k: find(k) if k else None)

root_keys = collections.defaultdict(collections.Counter)
for r in M[~M['ambiguous']].itertuples():
    root_keys[r.root][r.key] += 1


def entity_id(root):
    """덩어리 대표에서 기업 ID를 정한다."""
    if kr_of.get(root):
        return kr_of[root]
    keys = root_keys.get(root) or collections.Counter({root: 1})
    en = [(n, k) for k, n in keys.items() if k.startswith('en:')]
    if en:
        return 'EN:' + max(en)[1][3:]
    return 'KO:' + keys.most_common(1)[0][0][3:]


M['entity_id'] = [None if amb or not root else entity_id(root) for amb, root in zip(M['ambiguous'], M['root'])]


def how(r):
    """언급이 어떻게 이어졌는지(상장사 / 이름 / 애매)."""
    if r.ambiguous:
        return '애매(그룹 이름)'
    if r.entity_id and r.entity_id.startswith('KR:'):
        return '상장사'
    return '이름'


M['how'] = [how(r) for r in M.itertuples()]

# 기업 표
co = companies.set_index('stock_code')
ent_rows = []
for eid, g in M[M['entity_id'].notna()].groupby('entity_id'):
    names_en = collections.Counter(n for n in g['name_en'] if n) + collections.Counter(n for n in g['name'] if n and not HANGUL.search(n))
    names_ko = collections.Counter(n for n in g['name'] if HANGUL.search(n))
    countries = collections.Counter(c for c in g['country'] if c)
    row = dict(entity_id=eid, stock_code='', name_en=names_en.most_common(1)[0][0] if names_en else '',
               name_ko=names_ko.most_common(1)[0][0] if names_ko else '', country=countries.most_common(1)[0][0] if countries else '',
               n_news=int((g['source'] == 'news').sum()), n_dart=int((g['source'] != 'news').sum()),
               other_names=' | '.join(n for n, _ in (names_en + names_ko).most_common(6)))
    if eid.startswith('KR:') and eid[3:] in co.index:
        c = co.loc[eid[3:]]
        row.update(stock_code=eid[3:], name_en=c['corp_name_eng'], name_ko=c['stock_name'], country='South Korea')
    elif eid.startswith('KR:'):                      # 후보 밖 상장사: sector.json 종목명, DART 고유번호 목록의 영어 이름
        row.update(stock_code=eid[3:], name_en=corp_by_stock.get(eid[3:], ('', ''))[1], name_ko=OUTSIDE[eid[3:]], country='South Korea')
    ent_rows.append(row)
# 언급이 없는 서비스 대상 후보도 노드로 둔다
seen = {r['entity_id'] for r in ent_rows}
for r in companies.itertuples():
    if 'KR:' + r.stock_code not in seen:
        ent_rows.append(dict(entity_id='KR:' + r.stock_code, stock_code=r.stock_code, name_en=r.corp_name_eng, name_ko=r.stock_name,
                             country='South Korea', n_news=0, n_dart=0, other_names=''))
E = pd.DataFrame(ent_rows).sort_values(['n_news', 'n_dart'], ascending=False)
print(f'기업 노드 {len(E):,}개 (상장사 {E.entity_id.str.startswith("KR:").sum():,}, 영어 이름 {E.entity_id.str.startswith("EN:").sum():,}, '
      f'한글 이름만 {E.entity_id.str.startswith("KO:").sum():,})')

기업 노드 40,824개 (상장사 1,573, 영어 이름 18,838, 한글 이름만 20,413)


## 6. 저장

In [6]:
# ── 저장 ──
def write_if_changed(path, df):
    """내용이 바뀌었을 때만 CSV를 쓴다(iCloud 동기화 사본이 생기지 않게)."""
    content = '﻿' + df.to_csv(index=False)
    if path.exists() and path.read_text(encoding='utf-8') == content:
        return False
    path.write_text(content, encoding='utf-8')
    return True


cols = ['source', 'doc', 'field', 'ref', 'name', 'name_en', 'sure', 'country', 'entity_id', 'how']
w = [write_if_changed(OUT / 'companies.csv', E),
     write_if_changed(OUT / 'company_mentions.csv', M[cols]),
     write_if_changed(OUT / 'company_ambiguous.csv', M[M['ambiguous']][cols])]
print('저장', dict(zip(['companies', 'company_mentions', 'company_ambiguous'], w)))

저장 {'companies': True, 'company_mentions': True, 'company_ambiguous': False}


## 7. 결과 확인

1. 묶은 근거별 언급 수
2. 뉴스에 많이 나온 기업
3. 국내 상장사가 뉴스에서 잡힌 정도: 나라가 South Korea인 뉴스 기업 언급 중 상장사 ID가 붙은 비율과, 붙지 않은 이름
4. 여러 이름이 하나로 묶인 예
5. 묶지 않은 상장사 충돌

In [7]:
# ── 확인 1~2: 근거별 언급 수, 뉴스에 많이 나온 기업 ──
print('1) 묶은 근거별 언급 수')
display(M.groupby(['source', 'how']).size().unstack(fill_value=0))
print('2) 뉴스에 많이 나온 기업')
display(E.head(15)[['entity_id', 'name_en', 'name_ko', 'country', 'n_news', 'n_dart']])

1) 묶은 근거별 언급 수


how,상장사,애매(그룹 이름),이름
source,,,
dart_사업,2818,293,10073
dart_지분,2603,21,46508
news,112,22,8157


2) 뉴스에 많이 나온 기업


,entity_id,name_en,name_ko,country,n_news,n_dart
12319,EN:nvidia,Nvidia,,United States,146,5
5755,EN:google,Google,구글,,97,16
637,EN:amazon com,Amazon,아마존,,91,14
16337,EN:space exploration technologies,Space Exploration Technologies Corp.,,United States,83,0
11212,EN:microsoft,Microsoft,마이크로소프트,United States,79,16
11163,EN:meta platforms,"Meta Platforms, Inc.",메타 플랫폼스,,72,2
806,EN:anthropic pbc,Anthropic,,United States,71,1
12540,EN:openai,OpenAI,,United States,66,0
17195,EN:tesla,Tesla,테슬라,United States,53,10
859,EN:apple,Apple,애플,,42,17


In [8]:
# ── 확인 3: 뉴스의 국내 기업이 상장사로 이어졌는지 ──
print('3) 국내 기업(뉴스)')
kr = M[(M['source'] == 'news') & (M['country'] == 'South Korea')]
print(f"국내 기업 언급 {len(kr)}개 중 상장사 ID {kr['entity_id'].fillna('').str.startswith('KR:').sum()}개, 애매 {kr['ambiguous'].sum()}개")
display(kr[~kr['entity_id'].fillna('').str.startswith('KR:') & ~kr['ambiguous']]['name_en'].value_counts().head(20))

3) 국내 기업(뉴스)
국내 기업 언급 206개 중 상장사 ID 106개, 애매 22개


name_en
Samsung Display Co., Ltd.                11
Hyundai Motor Group                       4
Lotte Duty Free                           3
Coupang, Inc.                             2
Korea Hydro & Nuclear Power Co., Ltd.     2
Samsung Group                             2
Laneige                                   2
Woori Card Co., Ltd.                      1
IMM Private Equity                        1
Hyundai LNG Shipping                      1
CJ Olive Young                            1
Kyobo Book Centre                         1
Lotte Card Co., Ltd.                      1
Upstage                                   1
RLWRLD                                    1
Point2                                    1
Rebellions                                1
IMM Investment                            1
Hanwha Qcells                             1
CJ Group                                  1
Name: count, dtype: int64

In [9]:
# ── 확인 4~5: 뉴스와 DART에 모두 나온 기업, 묶지 않은 상장사 충돌 ──
print('4) 여러 이름이 하나로 묶인 예(뉴스와 DART에 모두 나온 기업)')
both = E[(E['n_news'] > 0) & (E['n_dart'] > 0)]
print(f'뉴스와 DART에 모두 나온 기업 {len(both)}개')
display(both.head(20)[['entity_id', 'name_en', 'name_ko', 'n_news', 'n_dart', 'other_names']])
print('5) 묶지 않은 상장사 충돌')
display(pd.DataFrame(conflicts).head(20))

4) 여러 이름이 하나로 묶인 예(뉴스와 DART에 모두 나온 기업)
뉴스와 DART에 모두 나온 기업 410개


,entity_id,name_en,name_ko,n_news,n_dart,other_names
12319,EN:nvidia,Nvidia,,146,5,Nvidia | NVIDIA Corporation | Nvidia Corporation | Nvidi...
5755,EN:google,Google,구글,97,16,Google | Google LLC | 구글 | GOOGLE
637,EN:amazon com,Amazon,아마존,91,14,"Amazon | Amazon.com, Inc. | Amazon.com Inc. | Amazon.com..."
11212,EN:microsoft,Microsoft,마이크로소프트,79,16,Microsoft | Microsoft Corporation | Microsoft Corp. | 마이...
11163,EN:meta platforms,"Meta Platforms, Inc.",메타 플랫폼스,72,2,"Meta Platforms, Inc. | Meta | Meta Platforms | Meta Plat..."
806,EN:anthropic pbc,Anthropic,,71,1,Anthropic | Anthropic PBC
17195,EN:tesla,Tesla,테슬라,53,10,"Tesla | Tesla, Inc. | Tesla Inc. | 테슬라"
859,EN:apple,Apple,애플,42,17,Apple | Apple Inc. | 애플 | Apple Inc | Apple사
292,EN:advanced micro devices,"Advanced Micro Devices, Inc.",,40,2,"Advanced Micro Devices, Inc. | AMD | Advanced Micro Devi..."
576,EN:alphabet,Alphabet Inc.,,38,2,Alphabet Inc. | Alphabet | *ALPHABET


5) 묶지 않은 상장사 충돌


,a,b,kr_a,kr_b,why
0,ko:한국철강,en:korea steel,KR:104700,KR:007280,한 언급의 두 이름
1,ko:한국철강,en:korea steel,KR:104700,KR:007280,한 언급의 두 이름
2,ko:한국철강,en:korea steel,KR:104700,KR:007280,한 언급의 두 이름
3,ko:한국특강,en:hankook steel,KR:007280,KR:025890,한 언급의 두 이름
4,ko:한국파마,en:korea pharma,KR:032300,KR:014570,한 언급의 두 이름
